# Stage 1 Go/No-Go Experiment (PMB vs baselines under identity corruption)

Run the cells top to bottom (Runtime → Run all).

1. Install deps
2. Write the 7 Python files into a `stage1/` package (the code imports `stage1.xxx`)
3. Run the experiment on **synthetic, CICIoT2023-shaped data** (validates the pipeline only, not the real Stage 1 answer)
4. Optional: multi-seed run for a less noisy read

In [ ]:
!pip -q install numpy scipy scikit-learn pandas dpkt

## 1. Create the package

In [ ]:
import os
os.makedirs('stage1', exist_ok=True)
open('stage1/__init__.py','w').close()

In [ ]:
%%writefile stage1/corruption.py
# -*- coding: utf-8 -*-
"""
Identity corruption simulator for the Stage 1 go/no-go experiment.

Implements the 5 corruption knobs from the roadmap:
  - MAC rotation rate: device identifier changes over time
  - Hub/NAT aggregation size: N devices merged into one observed stream
  - Spoof-duplicate fraction: devices sharing an identifier
  - Hidden/unknown-device fraction: devices with no usable identifier
  - Fingerprinting error rate: identity-resolution stage assigns wrong device

Each knob is applied independently so you can vary them one at a time
(as the roadmap specifies) or combined.
"""
from dataclasses import dataclass
import numpy as np


@dataclass
class CorruptionConfig:
    mac_rotation_rate: float = 0.0       # prob. a device's ID changes at each window
    hub_aggregation_size: int = 1        # how many true devices share one observed ID
    spoof_duplicate_frac: float = 0.0    # fraction of devices that share an ID w/ another
    hidden_device_frac: float = 0.0      # fraction of devices with no usable ID (None)
    fingerprint_error_rate: float = 0.0  # used only by B1's resolver, not here


def apply_corruption(records, config: CorruptionConfig, rng: np.random.Generator):
    """
    records: list of dicts, each with at least:
        {'true_device_id': int, 'window': int, 'features': np.ndarray, 'is_attack': bool}
    Returns a NEW list of dicts with an added 'observed_id' field (str or None),
    'true_device_id' is preserved for evaluation only -- the detector/tracker
    must never be given 'true_device_id' directly.
    """
    n_true_devices = len({r['true_device_id'] for r in records})

    # --- hub aggregation: map groups of true devices to one hub id ---
    device_ids = sorted({r['true_device_id'] for r in records})
    hub_map = {}
    if config.hub_aggregation_size > 1:
        for i, dev in enumerate(device_ids):
            hub_map[dev] = f"hub_{i // config.hub_aggregation_size}"
    else:
        hub_map = {dev: f"dev_{dev}" for dev in device_ids}

    # --- spoof duplicates: some devices forced to share another device's id ---
    spoof_map = {}
    n_spoofed = int(len(device_ids) * config.spoof_duplicate_frac)
    spoof_targets = rng.choice(device_ids, size=n_spoofed, replace=False) if n_spoofed else []
    for dev in spoof_targets:
        victim = rng.choice([d for d in device_ids if d != dev])
        spoof_map[dev] = hub_map[victim]

    # --- hidden devices: never get a usable id ---
    n_hidden = int(len(device_ids) * config.hidden_device_frac)
    hidden_devices = set(rng.choice(device_ids, size=n_hidden, replace=False)) if n_hidden else set()

    # --- per-device rotating alias state for MAC rotation ---
    rotation_state = {dev: hub_map[dev] for dev in device_ids}

    out = []
    for r in records:
        dev = r['true_device_id']
        rec = dict(r)

        if dev in hidden_devices:
            rec['observed_id'] = None
            out.append(rec)
            continue

        # MAC rotation: with given probability, this device gets a fresh alias
        if config.mac_rotation_rate > 0 and rng.random() < config.mac_rotation_rate:
            rotation_state[dev] = f"{hub_map[dev]}_rot{rng.integers(0, 1_000_000)}"

        observed = rotation_state[dev]

        # spoof duplication overrides with the shared/victim id
        if dev in spoof_map:
            observed = spoof_map[dev]

        rec['observed_id'] = observed
        out.append(rec)

    return out

In [ ]:
%%writefile stage1/detector.py
# -*- coding: utf-8 -*-
"""
A lightweight, swappable anomaly-score detector.

Stage 1 is testing whether identity-aware tracking (P1) recovers performance
lost to identity corruption -- NOT testing which detector is best. So this
file deliberately uses a simple, fast, well-understood scorer (IsolationForest
on per-window features) as a stand-in for the real Paper-1 GRU/POMDP detector.

To use your real detector later: replace `score_windows()` with a call into
your trained GRU model, keeping the same input/output shape (features in,
one anomaly score in [0, 1] out per window). Nothing else in Stage 1 needs
to change.
"""
import numpy as np
from sklearn.ensemble import IsolationForest


def fit_detector(feature_matrix: np.ndarray, random_state=0):
    """Fits on a feature matrix assumed to be mostly-normal traffic."""
    clf = IsolationForest(
        n_estimators=150,
        contamination="auto",
        random_state=random_state,
    )
    clf.fit(feature_matrix)
    return clf


def score_windows(clf, feature_matrix: np.ndarray) -> np.ndarray:
    """
    Returns anomaly scores in [0, 1], higher = more anomalous.
    IsolationForest's decision_function is higher-for-normal, so we flip and
    min-max normalize for a consistent 0..1 anomaly convention.
    """
    raw = clf.decision_function(feature_matrix)  # higher = more normal
    inv = -raw
    lo, hi = inv.min(), inv.max()
    if hi - lo < 1e-12:
        return np.zeros_like(inv)
    return (inv - lo) / (hi - lo)


def fingerprint_resolve(records, error_rate: float, rng: np.random.Generator):
    """
    Simulates B1's identity-resolution stage: takes corrupted observed_ids
    and produces a "resolved" id per record, imperfectly. A correct resolver
    would recover the rotated/hub-merged identity; `error_rate` is the
    fraction of records where it fails and assigns a wrong/fresh label
    instead, modeling real fingerprinting error.
    """
    out = []
    for r in records:
        rec = dict(r)
        obs = rec['observed_id']
        if obs is None:
            rec['resolved_id'] = f"unresolved_{rng.integers(0, 1_000_000)}"
        elif rng.random() < error_rate:
            rec['resolved_id'] = f"misresolved_{rng.integers(0, 1_000_000)}"
        else:
            rec['resolved_id'] = obs
        out.append(rec)
    return out

In [ ]:
%%writefile stage1/baselines.py
# -*- coding: utf-8 -*-
"""
B0-B3 baselines from the roadmap's Stage 1 methods table.

B0: oracle (ground-truth) IDs, upper bound.
B1: identity-first (fingerprint/cluster to hard IDs, then per-device).
B2: network-level/aggregate, identity-free.
B3: independent per-device Bernoulli on the (possibly wrong) raw observed IDs
    -- naive, no birth model, no clutter handling. Tests whether the PMB's
    extra machinery is needed at all, or independent-per-id tracking is enough.
"""
from collections import defaultdict


def run_b0_oracle(records_scored):
    """records_scored: list of dicts with true_device_id and anomaly_score.
    Returns {true_device_id: set of windows flagged compromised}."""
    flagged = defaultdict(set)
    for r in records_scored:
        if r['anomaly_score'] >= 0.5:
            flagged[r['true_device_id']].add(r['window'])
    return flagged


def run_b1_identity_first(records_scored, threshold=0.5):
    """Uses resolved_id (from detector.fingerprint_resolve)."""
    flagged = defaultdict(set)
    for r in records_scored:
        if r['anomaly_score'] >= threshold:
            flagged[r['resolved_id']].add(r['window'])
    return flagged


def run_b2_aggregate(records_scored, threshold=0.5):
    """No identity at all -- one global stream. Returns just the set of
    windows where *something* looked compromised (no per-device breakdown
    possible by construction, which is exactly B2's limitation)."""
    flagged_windows = {r['window'] for r in records_scored if r['anomaly_score'] >= threshold}
    return flagged_windows


def run_b3_naive_per_id(records_scored, threshold=0.5, prior=0.1, update_gain=0.4):
    """
    FIX 3: independent per-id Bernoulli belief, matching the roadmap's B3 definition
    ("independent per-device Bernoulli beliefs on the (possibly wrong) IDs").

    Each raw observed_id gets its own belief p = P(compromised), initialised at `prior`
    and updated recursively whenever that id is observed (same smoothing gain as P1's
    update_gain, so the comparison isolates the missing machinery, not tuning):
        p <- (1 - g) * p + g * anomaly_score
    An id is flagged in a window if it is observed in that window and p >= threshold.

    What B3 deliberately LACKS (this is what P1 adds):
      - no Poisson pool / birth model (every id gets a belief immediately, clutter included)
      - no existence probability, no confirmation-hits requirement
      - no decay/death: beliefs never fade when an id goes unseen
    Ids that are None (hidden devices) are skipped.
    """
    flagged = defaultdict(set)
    belief = {}
    for r in sorted(records_scored, key=lambda x: x['window']):
        oid = r['observed_id']
        if oid is None:
            continue
        p = belief.get(oid, prior)
        p = (1 - update_gain) * p + update_gain * r['anomaly_score']
        belief[oid] = p
        if p >= threshold:
            flagged[oid].add(r['window'])
    return flagged

In [ ]:
%%writefile stage1/pmb_filter.py
# -*- coding: utf-8 -*-
"""
Simplified, single-hypothesis Poisson Multi-Bernoulli (PMB) filter -- the
P1 / proposed method from the roadmap.

This is deliberately a SIMPLIFIED PMB, not a full PMBM (no multi-hypothesis
data association). It matches the structural choice noted in Update 5:
an explicit Poisson pool for "not yet confirmed" observations, and a
confirmed Bernoulli track is only spawned once evidence (repeated
observation + anomaly score) justifies it -- rather than pre-populating
every possible id with existence probability from step one (which Update 5
identified as LMB's likely weakness in the earlier unfair comparison).

Honest scope note: this omits full RFS machinery (no OSPA-optimal
multi-hypothesis association, no proper Bayes recursion over state
densities). It's built to test the GO/NO-GO question in the roadmap --
does *tracking identity probabilistically* recover performance lost to
corruption -- not to be the final, paper-ready filter. If Stage 1 passes,
upgrading this to a real PMBM implementation is the next step, per the
roadmap's own "planned upgrade path."
"""
from dataclasses import dataclass, field
import numpy as np


@dataclass
class Track:
    label: str
    r: float                 # existence probability
    compromise_belief: float # P(this device is compromised)
    last_seen: int
    born_at: int
    hits: int = 1


@dataclass
class PMBConfig:
    birth_threshold: float = 0.5     # anomaly score needed to seed a new track from the Poisson pool
    birth_r: float = 0.3             # initial existence prob for a newly spawned track
    confirm_hits: int = 2            # observations needed before a track counts as "confirmed"
    decay_per_window: float = 0.15   # existence prob decays this much per window unseen
    death_threshold: float = 0.05    # track removed once r drops below this
    update_gain: float = 0.4         # how much one observation moves compromise_belief


class PMBFilter:
    def __init__(self, config: PMBConfig):
        self.cfg = config
        self.tracks: dict[str, Track] = {}

    def step(self, window: int, observations: list[tuple[str, float]]):
        """
        observations: list of (observed_id, anomaly_score) seen in this window.
        observed_id may repeat within a window if hub-aggregated; that's fine,
        each is processed independently, matching normal RFS measurement handling.
        """
        seen_this_window = set()

        for obs_id, score in observations:
            seen_this_window.add(obs_id)
            if obs_id in self.tracks:
                t = self.tracks[obs_id]
                t.hits += 1
                t.last_seen = window
                t.r = min(1.0, t.r + (1 - t.r) * 0.5)  # re-observation raises existence confidence
                t.compromise_belief = (
                    (1 - self.cfg.update_gain) * t.compromise_belief
                    + self.cfg.update_gain * score
                )
            else:
                # Poisson pool decision: does this look enough like a real,
                # noteworthy object to spawn a confirmed-candidate Bernoulli track?
                if score >= self.cfg.birth_threshold:
                    self.tracks[obs_id] = Track(
                        label=obs_id,
                        r=self.cfg.birth_r,
                        compromise_belief=score,
                        last_seen=window,
                        born_at=window,
                    )
                # else: treated as clutter, discarded, no track created

        # decay / death for tracks not seen this window
        dead = []
        for label, t in self.tracks.items():
            if label not in seen_this_window:
                t.r -= self.cfg.decay_per_window
                if t.r < self.cfg.death_threshold:
                    dead.append(label)
        for label in dead:
            del self.tracks[label]

    def confirmed_compromised(self, compromise_threshold: float = 0.5):
        """Returns the set of track labels currently believed compromised."""
        return {
            t.label for t in self.tracks.values()
            if t.hits >= self.cfg.confirm_hits and t.compromise_belief >= compromise_threshold
        }

    def cardinality_estimate(self):
        """Expected number of existing objects = sum of existence probabilities."""
        return sum(t.r for t in self.tracks.values())

In [ ]:
%%writefile stage1/metrics.py
# -*- coding: utf-8 -*-
"""
Metrics for the Stage 1 go/no-go experiment.

Implements: cardinality error, per-window detection F1 (mapped back to true
devices via majority vote, since observed/resolved labels aren't the true
device id), and false isolation rate. A full OSPA metric is NOT implemented
here (noted in the roadmap as still needing metric-choice confirmation) --
label_to_true_device() below is the piece a real OSPA implementation would
also need, so it's built to be reused.
"""
from collections import defaultdict, Counter
import numpy as np


def label_to_true_device_map(records):
    """
    Builds {observed_or_resolved_label: majority true_device_id it maps to}.
    This uses ground truth and must ONLY be used for evaluation, never fed
    to a detector/tracker.
    """
    votes = defaultdict(Counter)
    for r in records:
        for key in ('observed_id', 'resolved_id'):
            if key in r and r[key] is not None:
                votes[r[key]][r['true_device_id']] += 1
    return {label: counter.most_common(1)[0][0] for label, counter in votes.items()}


def cardinality_mae(estimated_per_window: dict, true_per_window: dict):
    windows = sorted(set(estimated_per_window) | set(true_per_window))
    errs = [abs(estimated_per_window.get(w, 0) - true_per_window.get(w, 0)) for w in windows]
    return float(np.mean(errs)) if errs else float('nan')


def true_compromised_per_window(records):
    """{window: set(true_device_id) that are actually compromised in that window}."""
    out = defaultdict(set)
    for r in records:
        if r['is_attack']:
            out[r['window']].add(r['true_device_id'])
    return out


def detection_prf1(flagged_by_label: dict, id_map: dict, true_per_window: dict, clean_devices: set):
    """
    flagged_by_label: {label: set(windows flagged)}  -- from a baseline/P1 run
    id_map: label -> true_device_id (majority vote, eval-only)
    true_per_window: {window: set(true compromised device ids)}
    clean_devices: set of true_device_id that are NEVER actually compromised
                   (used for false isolation rate)

    Returns dict with precision, recall, f1, false_isolation_rate.
    """
    tp = fp = fn = 0
    false_isolations = 0
    all_windows = sorted(true_per_window)

    for w in all_windows:
        predicted_devices = {
            id_map.get(label, label)  # fall back to raw label if unmapped
            for label, windows in flagged_by_label.items() if w in windows
        }
        true_devices = true_per_window.get(w, set())

        tp += len(predicted_devices & true_devices)
        fp += len(predicted_devices - true_devices)
        fn += len(true_devices - predicted_devices)
        false_isolations += len(predicted_devices & clean_devices)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    fir = false_isolations / (len(clean_devices) * len(all_windows)) if clean_devices and all_windows else 0.0

    return {
        'precision': precision, 'recall': recall, 'f1': f1,
        'false_isolation_rate': fir,
    }

In [ ]:
%%writefile stage1/data.py
# -*- coding: utf-8 -*-
"""
Data loading for Stage 1.

generate_synthetic()   CICIoT2023-SHAPED toy data (validates the pipeline only).
inspect_ciciot2023()   RUN THIS FIRST on your real data: tells you what you actually have.
load_ciciot2023()      Real loader. Returns the same record shape as generate_synthetic():
                         {'true_device_id': int, 'window': int, 'features': np.ndarray, 'is_attack': bool}
                       Two modes (auto-detected):
                         'csv'  -- CSV files that contain a device-identifier column
                         'pcap' -- raw pcaps; builds per-source-MAC time-window features itself

IMPORTANT (from the CICIoT2023 docs as I know them, verify with inspect_ciciot2023):
the pre-extracted CSVs (46 features + label) carry NO device/MAC/IP column, so they
cannot give per-device ground truth. Stage 1 needs ground-truth device identity, hence
the pcap mode.
"""
import os
import re
import glob
import numpy as np


# ----------------------------------------------------------------------------- synthetic
def generate_synthetic(
    n_devices=40,
    n_windows=200,
    n_features=8,
    attack_devices_frac=0.25,
    attack_start_frac=0.5,
    seed=0,
):
    """Toy dataset with the same record shape as the real loader output."""
    rng = np.random.default_rng(seed)
    n_attack_devices = int(n_devices * attack_devices_frac)
    attack_devices = set(rng.choice(n_devices, size=n_attack_devices, replace=False).tolist())
    attack_start_window = int(n_windows * attack_start_frac)

    records = []
    for dev in range(n_devices):
        base = rng.normal(0, 1, size=n_features)
        for w in range(n_windows):
            is_attack = dev in attack_devices and w >= attack_start_window
            if is_attack:
                feat = base + rng.normal(3.0, 0.8, size=n_features)
            else:
                feat = base + rng.normal(0, 0.5, size=n_features)
            records.append({'true_device_id': dev, 'window': w, 'features': feat,
                            'is_attack': bool(is_attack)})
    return records


# ----------------------------------------------------------------------------- inspection
_ID_PAT = re.compile(r'(mac|device|dev_id|src.?ip|source.?ip|src_addr|eth.?src|host)', re.I)


def _find_files(path, exts):
    if os.path.isfile(path):
        return [path] if path.lower().endswith(exts) else []
    out = []
    for ext in exts:
        out += glob.glob(os.path.join(path, '**', '*' + ext), recursive=True)
    return sorted(set(out))


def inspect_ciciot2023(path, n_rows=50000):
    """Prints what is in `path`: file counts, CSV columns, any identity-like columns, label values."""
    import pandas as pd
    csvs = _find_files(path, ('.csv',))
    pcaps = _find_files(path, ('.pcap', '.pcapng'))
    gb = lambda fs: sum(os.path.getsize(f) for f in fs) / 1e9
    print(f"path: {path}")
    print(f"  CSV files : {len(csvs)}  ({gb(csvs):.2f} GB)")
    print(f"  pcap files: {len(pcaps)}  ({gb(pcaps):.2f} GB)")
    info = {'csv_has_id': False, 'id_columns': [], 'n_csv': len(csvs), 'n_pcap': len(pcaps)}
    if csvs:
        df = pd.read_csv(csvs[0], nrows=n_rows)
        print(f"\nFirst CSV: {os.path.basename(csvs[0])}  shape(read)={df.shape}")
        print("  columns:", list(df.columns))
        ids = [c for c in df.columns if _ID_PAT.search(str(c))]
        info['id_columns'] = ids
        info['csv_has_id'] = bool(ids)
        print("  identity-like columns:", ids if ids else "NONE")
        lab = [c for c in df.columns if str(c).lower() == 'label']
        if lab:
            print("  label values:", df[lab[0]].value_counts().head(10).to_dict())
    if pcaps:
        print("\nExample pcap names:", [os.path.basename(p) for p in pcaps[:8]])
    if info['csv_has_id']:
        rec = "csv (identity column found)"
    elif pcaps:
        rec = "pcap (no identity column in CSVs; pcaps will give per-source-MAC ground truth)"
    elif csvs:
        rec = ("NONE USABLE -- CSVs have no device identity column and no pcaps were found. "
               "Download the CICIoT2023 pcaps, or use a dataset that ships device IDs.")
    else:
        rec = "NONE -- no .csv or .pcap files found under this path"
    print("\nRECOMMENDED MODE:", rec)
    return info


# ----------------------------------------------------------------------------- CSV mode
def _load_csv_mode(path, id_col, label_col, benign_label, max_rows_per_file, max_devices):
    import pandas as pd
    files = _find_files(path, ('.csv',))
    frames = []
    for f in files:
        frames.append(pd.read_csv(f, nrows=max_rows_per_file))
    df = pd.concat(frames, ignore_index=True)
    if id_col is None:
        cands = [c for c in df.columns if _ID_PAT.search(str(c))]
        if not cands:
            raise ValueError("No identity-like column in the CSVs. CICIoT2023's pre-extracted CSVs have none; "
                             "use pcap mode (mode='pcap') instead.")
        id_col = cands[0]
        print(f"[csv mode] using identity column: {id_col}")
    if label_col is None:
        label_col = next(c for c in df.columns if str(c).lower() == 'label')
    feat_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in (id_col, label_col)]
    dev_codes, dev_index = np.unique(df[id_col].astype(str).values, return_inverse=True)
    keep = np.arange(len(dev_codes))
    if max_devices and len(keep) > max_devices:
        keep = np.arange(max_devices)
    df = df.assign(_dev=dev_index)
    df = df[df['_dev'].isin(keep)]
    df['_win'] = df.groupby('_dev').cumcount()          # row order within a device = its window index
    records = []
    X = df[feat_cols].to_numpy(dtype=float)
    is_att = (df[label_col].astype(str).str.lower() != benign_label.lower()).to_numpy()
    for x, d, w, a in zip(X, df['_dev'].to_numpy(), df['_win'].to_numpy(), is_att):
        records.append({'true_device_id': int(d), 'window': int(w), 'features': x, 'is_attack': bool(a)})
    return records


# ----------------------------------------------------------------------------- pcap mode
class _Acc:
    __slots__ = ('n', 'bytes', 'bytes2', 'tcp', 'udp', 'icmp', 'syn', 'ack', 'rst',
                 'dns', 'http', 'https', 'dst_ips', 'dst_ports', 'first', 'last')

    def __init__(self):
        self.n = self.bytes = self.bytes2 = 0
        self.tcp = self.udp = self.icmp = self.syn = self.ack = self.rst = 0
        self.dns = self.http = self.https = 0
        self.dst_ips, self.dst_ports = set(), set()
        self.first = self.last = None

    def vector(self, window_seconds):
        n = max(self.n, 1)
        mean_len = self.bytes / n
        std_len = max(self.bytes2 / n - mean_len ** 2, 0.0) ** 0.5
        span = max((self.last - self.first) if self.first is not None else 0.0, 1e-3)
        return np.array([
            self.n / window_seconds, self.bytes / window_seconds, mean_len, std_len,
            self.tcp / n, self.udp / n, self.icmp / n, self.syn / n, self.ack / n, self.rst / n,
            len(self.dst_ips), len(self.dst_ports), self.dns / n, self.http / n, self.https / n,
            self.n / span,
        ], dtype=float)


PCAP_FEATURE_NAMES = ['pkts_per_s', 'bytes_per_s', 'mean_len', 'std_len', 'frac_tcp', 'frac_udp',
                      'frac_icmp', 'frac_syn', 'frac_ack', 'frac_rst', 'uniq_dst_ip', 'uniq_dst_port',
                      'frac_dns', 'frac_http', 'frac_https', 'burst_rate']


def _pcap_reader(fh):
    import dpkt
    try:
        return dpkt.pcap.Reader(fh)
    except (ValueError, dpkt.dpkt.NeedData):
        fh.seek(0)
        return dpkt.pcapng.Reader(fh)


def _extract_pcap(path, window_seconds, max_packets):
    """Returns {(src_mac_hex, window_idx): _Acc} and number of windows in this pcap."""
    import dpkt
    accs, t0, max_w = {}, None, -1
    with open(path, 'rb') as fh:
        reader = _pcap_reader(fh)
        for i, (ts, buf) in enumerate(reader):
            if max_packets and i >= max_packets:
                break
            try:
                eth = dpkt.ethernet.Ethernet(buf)
            except Exception:
                continue
            if t0 is None:
                t0 = ts
            w = int((ts - t0) // window_seconds)
            max_w = max(max_w, w)
            a = accs.setdefault((eth.src.hex(), w), _Acc())
            a.n += 1
            a.bytes += len(buf)
            a.bytes2 += len(buf) ** 2
            if a.first is None:
                a.first = ts
            a.last = ts
            ip = eth.data
            if isinstance(ip, dpkt.ip.IP):
                a.dst_ips.add(ip.dst)
                l4 = ip.data
                if isinstance(l4, dpkt.tcp.TCP):
                    a.tcp += 1
                    a.syn += bool(l4.flags & dpkt.tcp.TH_SYN)
                    a.ack += bool(l4.flags & dpkt.tcp.TH_ACK)
                    a.rst += bool(l4.flags & dpkt.tcp.TH_RST)
                    a.dst_ports.add(l4.dport)
                    a.http += l4.dport in (80, 8080)
                    a.https += l4.dport == 443
                elif isinstance(l4, dpkt.udp.UDP):
                    a.udp += 1
                    a.dst_ports.add(l4.dport)
                    a.dns += l4.dport == 53
                elif isinstance(l4, dpkt.icmp.ICMP):
                    a.icmp += 1
    return accs, max_w + 1


def _load_pcap_mode(path, window_seconds, benign_max_files, attack_max_files, max_packets_per_file,
                    min_pkts_per_window, attacker_macs, exclude_macs, max_devices):
    pcaps = _find_files(path, ('.pcap', '.pcapng'))
    if not pcaps:
        raise FileNotFoundError(f"No .pcap/.pcapng files under {path}")
    is_benign = lambda p: 'benign' in os.path.basename(p).lower()
    benign = [p for p in pcaps if is_benign(p)][:benign_max_files]
    attack = [p for p in pcaps if not is_benign(p)][:attack_max_files]
    if not benign:
        raise ValueError("No benign pcap (filename containing 'benign') found. The detector is fitted on "
                         "the earliest windows, which must be benign.")
    attacker_macs = {m.lower().replace(':', '') for m in (attacker_macs or [])}
    exclude_macs = {m.lower().replace(':', '') for m in (exclude_macs or [])}

    raw, offset, benign_macs = [], 0, set()
    for is_att_file, files in ((False, benign), (True, attack)):     # benign FIRST -> early windows are normal
        for p in files:
            print(f"  reading {os.path.basename(p)} ({'attack' if is_att_file else 'benign'})")
            accs, nw = _extract_pcap(p, window_seconds, max_packets_per_file)
            for (mac, w), a in accs.items():
                if mac in exclude_macs or a.n < min_pkts_per_window:
                    continue
                if not is_att_file:
                    benign_macs.add(mac)
                # attack label: attack pcap AND (mac is a declared attacker OR mac never seen as a benign device)
                att = is_att_file and (mac in attacker_macs or mac not in benign_macs)
                raw.append((mac, offset + w, a.vector(window_seconds), att))
            offset += nw

    macs = sorted({r[0] for r in raw})
    if max_devices and len(macs) > max_devices:                       # keep the most active devices
        counts = {}
        for r in raw:
            counts[r[0]] = counts.get(r[0], 0) + 1
        macs = sorted(sorted(macs, key=lambda m: -counts[m])[:max_devices])
    idx = {m: i for i, m in enumerate(macs)}
    records = [{'true_device_id': idx[m], 'window': int(w), 'features': f, 'is_attack': bool(a)}
               for (m, w, f, a) in raw if m in idx]
    if benign_macs:
        print(f"  benign device MACs: {len(benign_macs)} | total devices kept: {len(macs)} | records: {len(records)}")
    first_att = min((r['window'] for r in records if r['is_attack']), default=None)
    n_win = max(r['window'] for r in records) + 1
    if first_att is not None and first_att < 0.25 * n_win:
        print("  WARNING: attack windows start inside the first 25% of time; the detector's fit slice "
              "would contain attacks. Load more benign files or fewer attack files.")
    return records


# ----------------------------------------------------------------------------- public loader
def load_ciciot2023(path, mode='auto', *, id_col=None, label_col=None, benign_label='BenignTraffic',
                    max_rows_per_file=200_000, max_devices=120,
                    window_seconds=5.0, benign_max_files=5, attack_max_files=10,
                    max_packets_per_file=500_000, min_pkts_per_window=3,
                    attacker_macs=None, exclude_macs=None):
    """
    mode: 'auto' | 'csv' | 'pcap'.  Auto = csv if an identity column exists, else pcap.

    Label semantics (pcap mode -- a DECISION you must own, see roadmap Update 6):
      benign pcaps            -> every source MAC is a device, is_attack=False
      attack pcaps            -> windows from a source MAC that was never seen in the benign pcaps
                                 (i.e. attacker hosts), or that you list in `attacker_macs`, are is_attack=True.
      Mirai-style cases where IoT devices themselves attack need `attacker_macs`.
    exclude_macs: e.g. the gateway/router MAC, if you don't want it treated as a device.
    """
    csvs = _find_files(path, ('.csv',))
    pcaps = _find_files(path, ('.pcap', '.pcapng'))
    if mode == 'auto':
        has_id = False
        if csvs:
            import pandas as pd
            cols = pd.read_csv(csvs[0], nrows=1).columns
            has_id = id_col is not None or any(_ID_PAT.search(str(c)) for c in cols)
        mode = 'csv' if has_id else ('pcap' if pcaps else 'csv')
        print(f"[auto] mode -> {mode}")
    if mode == 'csv':
        return _load_csv_mode(path, id_col, label_col, benign_label, max_rows_per_file, max_devices)
    if mode == 'pcap':
        return _load_pcap_mode(path, window_seconds, benign_max_files, attack_max_files,
                               max_packets_per_file, min_pkts_per_window, attacker_macs, exclude_macs, max_devices)
    raise ValueError("mode must be 'auto', 'csv' or 'pcap'")

In [ ]:
%%writefile stage1/run_stage1.py
# -*- coding: utf-8 -*-
"""
Stage 1 go/no-go experiment runner.

Usage:
    python run_stage1.py                 # dry run on synthetic data (validates the pipeline)
    python run_stage1.py --real PATH     # once load_ciciot2023() in data.py is filled in

This implements the comparison table from the roadmap:
    B0 = oracle ids (upper bound)
    B1 = identity-first (fingerprint/cluster, then per-device)
    B2 = aggregate, identity-free
    B3 = naive independent per-id Bernoulli (no birth model)
    P1 = PMB filter (Poisson pool + confirmed Bernoulli tracks) -- proposed method

Applies the roadmap's own pre-registered decision rule at the end:
    GO         if B1 loses a lot of B0's performance AND P1 recovers >= ~half the gap
               AND P1 clearly beats B3
    DOWNGRADE  if B3 matches P1 (identity ambiguity real, but PMB machinery adds nothing)
    NO-GO      if B1 degrades only gracefully (problem not severe enough)

These thresholds are editable at the top of main() -- the roadmap says to
set them before looking at results, so do that before running on real data.
"""
import argparse
import numpy as np
from collections import defaultdict

from stage1.corruption import CorruptionConfig, apply_corruption
from stage1.detector import fit_detector, score_windows, fingerprint_resolve
from stage1 import baselines
from stage1.pmb_filter import PMBFilter, PMBConfig
from stage1 import metrics
from stage1.data import generate_synthetic


def run_one_condition(clean_records, corruption_cfg: CorruptionConfig, rng, fp_error_rate: float,
                      fit_time_fraction: float = 0.25):
    # 1. corrupt identities
    corrupted = apply_corruption(clean_records, corruption_cfg, rng)

    # 2. fit + score with the (swappable) detector, using only normal-looking
    #    early windows to fit, matching realistic "train on mostly-normal" setup
    feat_matrix = np.array([r['features'] for r in corrupted])
    # FIX 1: fit on the EARLIEST windows (by time), not the first rows of the list.
    # Records are ordered device-by-device, so slicing by row index trained on a
    # handful of devices (including attack windows). Time-based split = train on
    # early, mostly-normal traffic, as intended.
    windows = np.array([r['window'] for r in corrupted])
    fit_cutoff = np.quantile(windows, fit_time_fraction)
    fit_mask = windows <= fit_cutoff
    clf = fit_detector(feat_matrix[fit_mask])
    scores = score_windows(clf, feat_matrix)
    for r, s in zip(corrupted, scores):
        r['anomaly_score'] = float(s)

    # 3. B1 needs a resolved id (imperfect fingerprinting)
    resolved = fingerprint_resolve(corrupted, error_rate=fp_error_rate, rng=rng)

    # 4. run baselines
    b0 = baselines.run_b0_oracle(resolved)
    b1 = baselines.run_b1_identity_first(resolved)
    b3 = baselines.run_b3_naive_per_id(resolved)

    # 5. run P1 (PMB) window by window, in time order
    pmb = PMBFilter(PMBConfig())
    by_window = defaultdict(list)
    for r in resolved:
        if r['observed_id'] is not None:
            by_window[r['window']].append((r['observed_id'], r['anomaly_score']))
    est_cardinality_per_window = {}
    p1_flagged = defaultdict(set)
    for w in sorted(by_window):
        pmb.step(w, by_window[w])
        est_cardinality_per_window[w] = pmb.cardinality_estimate()
        for label in pmb.confirmed_compromised():
            p1_flagged[label].add(w)

    # 6. evaluation setup
    id_map = metrics.label_to_true_device_map(resolved)
    true_pw = metrics.true_compromised_per_window(resolved)
    true_cardinality_pw = {w: len(devs) for w, devs in true_pw.items()}
    all_true_devices = {r['true_device_id'] for r in resolved}
    ever_compromised = {r['true_device_id'] for r in resolved if r['is_attack']}
    clean_devices = all_true_devices - ever_compromised

    def eval_baseline(flagged):
        return metrics.detection_prf1(flagged, id_map, true_pw, clean_devices)

    results = {
        'B0': eval_baseline(b0),
        'B1': eval_baseline(b1),
        'B3': eval_baseline(b3),
        'P1': eval_baseline(p1_flagged),
    }
    results['P1']['cardinality_mae'] = metrics.cardinality_mae(est_cardinality_per_window, true_cardinality_pw)

    return results


# --- decision rule thresholds: set BEFORE looking at real results (roadmap requirement) ---
B1_LOSS_FRACTION_NEEDED = 0.10   # B1 must lose >= 10% of B0's F1, else problem isn't severe (NO-GO)
RECOVERY_FRACTION_NEEDED = 0.5   # P1 must recover >= 50% of the B0-B1 gap
B3_MARGIN_NEEDED = 0.05          # P1's F1 must beat B3's F1 by at least this much


def decide(b0_f1, b1_f1, b3_f1, p1_f1):
    """
    Roadmap rule, evaluated in this order (FIX 2 -- NO-GO was previously unreachable
    whenever P1 ~ B3, because DOWNGRADE was checked first):
      1. NO-GO      if B1 degrades gracefully (loses < B1_LOSS_FRACTION_NEEDED of B0)
      2. DOWNGRADE  if B3 matches P1 (P1 - B3 < B3_MARGIN_NEEDED)
      3. GO         if P1 recovers >= RECOVERY_FRACTION_NEEDED of the gap
      4. otherwise  NO-GO for PMB (P1 recovers too little of the gap)
    Returns (verdict_string, details_dict).
    """
    gap = b0_f1 - b1_f1
    b1_loss = gap / b0_f1 if b0_f1 > 1e-9 else float('nan')
    recovered = (p1_f1 - b1_f1) / gap if gap > 1e-9 else float('nan')
    margin = p1_f1 - b3_f1
    details = {'gap': gap, 'b1_loss_fraction': b1_loss, 'recovered': recovered, 'margin_vs_b3': margin}

    if not (b1_loss >= B1_LOSS_FRACTION_NEEDED):
        verdict = "NO-GO (B1 degrades gracefully -- identity corruption not severe enough on this data)"
    elif margin < B3_MARGIN_NEEDED:
        verdict = "DOWNGRADE to Level 1 (B3 matches P1 -- PMB machinery not earning its keep)"
    elif recovered >= RECOVERY_FRACTION_NEEDED:
        verdict = "GO"
    else:
        verdict = "NO-GO for PMB (P1 recovers too little of the B0-B1 gap)"
    return verdict, details


def run_all(clean_records, seed=0, dry_run=True):
    """Runs the 3 corruption levels for all methods and applies decide(). Used by main() and notebooks."""
    rng = np.random.default_rng(seed)
    n_dev = len({r['true_device_id'] for r in clean_records})
    n_att = len({r['true_device_id'] for r in clean_records if r['is_attack']})
    print(f"{len(clean_records)} records | {n_dev} devices | {n_att} ever-attacking | "
          f"{'DRY RUN on synthetic data' if dry_run else 'REAL DATA'}")

    corruption_levels = [
        CorruptionConfig(mac_rotation_rate=0.0, hub_aggregation_size=1, spoof_duplicate_frac=0.0, hidden_device_frac=0.0),
        CorruptionConfig(mac_rotation_rate=0.15, hub_aggregation_size=1, spoof_duplicate_frac=0.1, hidden_device_frac=0.05),
        CorruptionConfig(mac_rotation_rate=0.4, hub_aggregation_size=3, spoof_duplicate_frac=0.25, hidden_device_frac=0.15),
    ]
    labels = ['clean (0.0)', 'moderate corruption', 'heavy corruption']

    print(f"\n{'condition':<22}{'method':<8}{'precision':<12}{'recall':<10}{'f1':<8}{'false_iso_rate':<16}")
    all_results = {}
    for label, cfg in zip(labels, corruption_levels):
        res = run_one_condition(clean_records, cfg, rng, fp_error_rate=0.15)
        all_results[label] = res
        for method, m in res.items():
            print(f"{label:<22}{method:<8}{m['precision']:<12.3f}{m['recall']:<10.3f}{m['f1']:<8.3f}{m['false_isolation_rate']:<16.3f}")

    heavy = all_results['heavy corruption']
    b0_f1, b1_f1, b3_f1, p1_f1 = heavy['B0']['f1'], heavy['B1']['f1'], heavy['B3']['f1'], heavy['P1']['f1']
    verdict, d = decide(b0_f1, b1_f1, b3_f1, p1_f1)

    print(f"\n--- Decision rule (heavy corruption condition) ---")
    print(f"B0={b0_f1:.3f}  B1={b1_f1:.3f}  B3={b3_f1:.3f}  P1={p1_f1:.3f}")
    print(f"B0-B1 gap: {d['gap']:.3f} ({d['b1_loss_fraction']:.1%} of B0) | "
          f"P1 recovers: {d['recovered']:.1%} of gap | P1 vs B3 margin: {d['margin_vs_b3']:.3f}")
    print(f"\nVERDICT: {verdict}")
    if dry_run:
        print("\n(This verdict is on SYNTHETIC data and only proves the pipeline runs end to end.)")
    else:
        print("\n(Caveat: B1's resolver is still a crude stand-in -- see roadmap Update 6.)")
    return all_results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--real', type=str, default=None, help='Path to CICIoT2023 (csv-with-ids or pcaps)')
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()
    if args.real:
        from stage1.data import load_ciciot2023
        run_all(load_ciciot2023(args.real), seed=args.seed, dry_run=False)
    else:
        run_all(generate_synthetic(seed=args.seed), seed=args.seed, dry_run=True)


if __name__ == '__main__':
    main()

## 2. Run the experiment (synthetic dry run)

In [ ]:
!python -m stage1.run_stage1 --seed 0

## 3. Multi-seed run (optional)
A single seed is noisy. This repeats the heavy-corruption condition over several seeds and reports mean ± std of F1 per method,
then applies the same `decide()` rule (defined in `stage1/run_stage1.py`) to the means.

In [ ]:
import numpy as np
from stage1.data import generate_synthetic
from stage1.corruption import CorruptionConfig
from stage1.run_stage1 import run_one_condition, decide

N_SEEDS = 10
heavy_cfg = CorruptionConfig(mac_rotation_rate=0.4, hub_aggregation_size=3,
                             spoof_duplicate_frac=0.25, hidden_device_frac=0.15)
f1 = {m: [] for m in ['B0', 'B1', 'B3', 'P1']}
for s in range(N_SEEDS):
    rng = np.random.default_rng(s)
    recs = generate_synthetic(seed=s)
    res = run_one_condition(recs, heavy_cfg, rng, fp_error_rate=0.15)
    for m in f1:
        f1[m].append(res[m]['f1'])

print(f"Heavy corruption, {N_SEEDS} seeds (F1 mean +/- std)")
for m, v in f1.items():
    print(f"  {m}: {np.mean(v):.3f} +/- {np.std(v):.3f}")

means = [np.mean(f1[m]) for m in ['B0', 'B1', 'B3', 'P1']]
verdict, d = decide(*means)
print(f"\nB0-B1 gap: {d['gap']:.3f} ({d['b1_loss_fraction']:.1%} of B0) | "
      f"P1 recovers {d['recovered']:.1%} of gap | P1 - B3 margin: {d['margin_vs_b3']:.3f}")
print(f"VERDICT (on seed-averaged means): {verdict}")

## 4. Real CICIoT2023 data
Steps: (a) point `DATA_PATH` at your CICIoT2023 folder (mount Drive if it lives there), (b) run **inspect** to see what you have,
(c) run **load + experiment**. Both cells skip cleanly if the path doesn't exist, so *Run all* never breaks.

Label semantics in pcap mode (you own this decision, see roadmap Update 6): windows from a source MAC never seen in the benign pcaps
(i.e. attacker hosts), or listed in `ATTACKER_MACS`, are `is_attack=True`. Put the gateway/router MAC in `EXCLUDE_MACS` if you don't want it treated as a device.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

DATA_PATH = "/content/drive/MyDrive/CICIoT2023"   # <-- EDIT THIS
ATTACKER_MACS = []    # e.g. ["aa:bb:cc:dd:ee:ff"]  (Mirai-style cases where IoT devices attack)
EXCLUDE_MACS  = []    # e.g. the gateway MAC

from stage1.data import inspect_ciciot2023
if os.path.exists(DATA_PATH):
    info = inspect_ciciot2023(DATA_PATH)
else:
    print("DATA_PATH not found -- edit it above, then re-run this cell.")

In [ ]:
from stage1.data import load_ciciot2023
from stage1.run_stage1 import run_all

if os.path.exists(DATA_PATH):
    records = load_ciciot2023(DATA_PATH, mode="auto",
                              window_seconds=5.0, benign_max_files=5, attack_max_files=10,
                              max_packets_per_file=500_000, max_devices=120,
                              attacker_macs=ATTACKER_MACS, exclude_macs=EXCLUDE_MACS)
    results = run_all(records, seed=0, dry_run=False)
else:
    print("Skipped: DATA_PATH not found.")

## Notes
- Sections 2-3 are synthetic (pipeline check only). Section 4 is the real-data path.
- Set your decision thresholds (`RECOVERY_FRACTION_NEEDED`, `B3_MARGIN_NEEDED` in `stage1/run_stage1.py`) **before** looking at real-data results.
- P1 here is a simplified single-hypothesis PMB, not full PMBM.
- B1's `fingerprint_resolve` is still a crude stand-in; treat any verdict as provisional until it's a real resolver.
- Changes vs. v1: detector now fits on early-time windows; decision rule order is NO-GO -> DOWNGRADE -> GO; B3 is a real per-id Bernoulli belief (smoothing, no birth model, no decay).